# Glaciers selection based on RGI6 

In [ ]:
# basic
from tqdm import tqdm
from pathlib import Path
import numpy as np
import pandas as pd


# spatial
import geopandas as gpd
import xarray as xr
import rioxarray as rio
import xrspatial as xrs
from exactextract import exact_extract
from src.utils import load_config, extract_timeseries, extract_attributes, climate_attributes

cfg = load_config()

In [ ]:
# basins (selection from basins_selection); the attributes this notebook derives are dropped,
# since the file is written back at the end and the zonal joins cannot overwrite columns
DERIVED = ["basin_area", "basin_cenlat", "basin_cenlon", "total_perim", "gc_index",
           "elev_mean", "elev_median", "elev_std", "slope_mean", "aspect_north", "aspect_east",
           "prcp_mean_ERA5", "prcp_pci_ERA5", "pet_mean_ERA5", "aridity_ERA5", "solid_prcp_ERA5",
           "frac_snow_ERA5", "low_prec_freq_ERA5", "high_prec_freq_ERA5", "pet_mean_GLEAM",
           "et_mean_GLEAM", "glacier_dhdt",
           "glacier_volume_M22", "glacier_volume_F19", "dams_count", "forest_cover",
           "shrubs_veg_cover", "herbaceous_veg_cover", "cropland_cover", "urban_cover",
           "sparse_veg_cover", "water_cover", "lai_max", "lai_diff"]
basins = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg")
basins = basins.drop(columns=DERIVED, errors="ignore")

## Dynamic inputs

In [ ]:
# Meteorological forcings (ERA5); precipitation from config.yml:climate.prcp_file (corrected)
variables = ["prcp", "tasmax", "tasmin", "tas", "ep"]

for variable in tqdm(variables):

    # Load only the yearly subset using time selection
    file_path = Path(cfg["dir"]["data"]) / f"era5/{variable}_ERA5_1960_2025.nc"
    if variable == "prcp":
        file_path = Path(cfg["dir"]["data"]) / cfg["climate"]["prcp_file"]

    with xr.open_dataset(file_path) as climate:
        climate = climate[variable].rename({"latitude": "lat", "longitude": "lon"})
        data = extract_timeseries(climate, basins, time_period=("1990-01-01", "2025-12-31"))
        data.to_parquet(f"data/climate/HydroAtlas_{variable}_ERA5.parquet", index=True)

## Static inputs
Same attributes, and the same definitions, as in AndeanGC v1.2

In [ ]:
# basic attributes
basins_projected = basins.to_crs(32719)  # UTM zone 19S for 
basins["basin_area"] = basins_projected.area / 1e6  # from m2 to km2
basins["basin_cenlat"] = basins_projected.centroid.to_crs(4326).y
basins["basin_cenlon"] = basins_projected.centroid.to_crs(4326).x
basins["total_perim"] = basins_projected.geometry.length / 1e3  # from m to km
basins["gc_index"] = basins.total_perim/(2*(np.sqrt(np.pi*basins.basin_area)))

In [ ]:
# elevation features (NASADEM)
dem = xr.open_dataset(cfg["dir"]["data"] + "/dem/DEM_Andes_270m.tif").band_data.sel(band=1, drop = True)
dem = dem.sel(x=slice(basins.total_bounds[0], basins.total_bounds[2]), 
              y=slice(basins.total_bounds[3], basins.total_bounds[1]))
dem  = dem.fillna(0)  # NAs to sea level (= 0)

slope  = xrs.slope(dem.rio.reproject("EPSG:32719"))
slope  = slope.rio.reproject("EPSG:4326")

# aspect as northness (cos) and eastness (sin), computed in UTM as in AndeanGC v1.2: a circular
aspect   = np.deg2rad(xrs.aspect(dem.rio.reproject("EPSG:32719")).where(lambda a: a >= 0))
northness = np.cos(aspect).rio.write_nodata(np.nan).rio.reproject("EPSG:4326")
eastness  = np.sin(aspect).rio.write_nodata(np.nan).rio.reproject("EPSG:4326")

basins = extract_attributes(dem, basins, "elev_mean", fun = "mean")
basins = extract_attributes(dem, basins, "elev_median", fun = "median")
basins = extract_attributes(dem, basins, "elev_std", fun = "stdev")
basins = extract_attributes(slope, basins, "slope_mean", fun = "mean")
basins = extract_attributes(northness, basins, "aspect_north", fun = "mean")
basins = extract_attributes(eastness, basins, "aspect_east", fun = "mean")

# basins that are entirely flat in the DEM (Lake Titicaca) have no aspect: no preferred exposure
basins[["aspect_north", "aspect_east"]] = basins[["aspect_north", "aspect_east"]].fillna(0)

In [ ]:
# climate attributes (ERA5, 1990-2019, AndeanGC v1.2 definitions); precipitation from the
# forcing file (config.yml:climate.prcp_file), so the statics match the dynamic inputs
era5_files = {v: Path(cfg["dir"]["data"]) / f"era5/{v}_ERA5_1960_2025.nc" for v in ["tasmax", "tasmin", "ep"]}
era5_files["prcp"] = Path(cfg["dir"]["data"]) / cfg["climate"]["prcp_file"]

climate_stack = xr.merge([xr.open_dataset(f, chunks={"time": 3653})[v] for v, f in era5_files.items()])
climate_stack = climate_stack.rename({"latitude": "lat", "longitude": "lon"})

basins = basins.join(climate_attributes(climate_stack, basins), on="gauge_id")

In [ ]:
# evaporation attributes (GLEAM v4.3a, 1990-2019, AndeanGC v1.2 definitions): annual totals in mm,
# averaged over the reference period. Renamed to x/y because GLEAM latitudes are descending.
gleam = xr.open_mfdataset(str(Path(cfg["dir"]["data"]) / "gleam" / "*_GLEAM_v4.3a_YR.nc"))
gleam = gleam.sel(time=slice("1990-01-01", "2019-12-31")).rename(lat="y", lon="x")
gleam = gleam.sel(x=slice(basins.total_bounds[0] - 1, basins.total_bounds[2] + 1),
                  y=slice(basins.total_bounds[3] + 1, basins.total_bounds[1] - 1)).mean("time").load()

basins = extract_attributes(gleam.Ep, basins, "pet_mean_GLEAM", fun="mean")  # potential evaporation
basins = extract_attributes(gleam.E, basins, "et_mean_GLEAM", fun="mean")    # actual evaporation

In [ ]:
# glacier attributes

# dhdt in m/yr
dhdt = rio.open_rasterio(cfg["dir"]["data"] + "/glacier_dhdt/dhdt_2000_2020_hugonnet.tif").sel(band=1, drop = True) # in m/yr
basins = extract_attributes(dhdt, basins, "glacier_dhdt", fun = "mean")

# vol in km3 (M22)
vol = rio.open_rasterio(cfg["dir"]["data"] + "/glacier_thickness/volume_millan_2022_100m.tif").sel(band=1, drop = True) # in km3
basins = extract_attributes(vol, basins, "glacier_volume_M22", fun = "sum")

# volume in km3 (F19)
vol = rio.open_rasterio(cfg["dir"]["data"] + "/glacier_thickness/volume_Farinotti_2019_200m.tif").sel(band=1, drop = True) # in km3
basins = extract_attributes(vol, basins, "glacier_volume_F19", fun = "sum")

In [ ]:
# water demand features
ref_period = ["1990-01-01", "2019-12-31"]

# dams 
dams_data = gpd.read_file(cfg["dir"]["data"] + "/dams/GDAT_V1_DAMS.gpkg") 
dams_data = dams_data[dams_data.Capac > 100] # only dams with capacity larger than 100 hm3

dams_count = gpd.sjoin(dams_data, basins.reset_index(), how='inner', predicate='within')
dams_count = dams_count.groupby('gauge_id').size()
# mapped on the gauge_id column: basins is not indexed by gauge_id until the land cover cell
basins["dams_count"] = basins.gauge_id.map(dams_count).fillna(0)

In [ ]:
# land cover features

land_cover_data = rio.open_rasterio(cfg["dir"]["data"] + "/land_cover/GLOBAL/PROBAV_LC100_global_v3.0.1_2019.tif").sel(band=1, drop = True)
land_cover_data = land_cover_data.sel(x=slice(basins.total_bounds[0], basins.total_bounds[2]), 
                                      y=slice(basins.total_bounds[3], basins.total_bounds[1]))

land_cover_data = land_cover_data.where(~((land_cover_data >= 100) & (land_cover_data <= 200)), 10)
land_cover_data = exact_extract(land_cover_data, basins, ["unique", "frac"], progress=False, 
                                output = "pandas", include_cols=["gauge_id"])

land_cover_data = land_cover_data.apply(lambda row: pd.Series(dict(zip(row['unique'], row['frac']))), axis=1)
land_cover_data = land_cover_data.set_index(basins.gauge_id).fillna(0) * 100

basins = basins.set_index("gauge_id")
basins["forest_cover"] = land_cover_data[10]
basins["shrubs_veg_cover"] = land_cover_data[20]
basins["herbaceous_veg_cover"] = land_cover_data[30]
basins["cropland_cover"] = land_cover_data[40]
basins["urban_cover"] = land_cover_data[50]
basins["sparse_veg_cover"] = land_cover_data[60]
basins["water_cover"] = land_cover_data[80]

In [ ]:
# vegetation features: LAI seasonality, as in ºAndeanGC v1.2

lai = xr.open_dataset(cfg["dir"]["data"] + "/lai/GIMMS_LAI4g_V1.2_SA_1982_2020_climatology.nc").lai
lai = lai.rename(lat="y", lon="x").sel(x=slice(basins.total_bounds[0], basins.total_bounds[2]),
                                       y=slice(basins.total_bounds[3], basins.total_bounds[1]))
lai = lai.fillna(0)

# one column per month, in the row order of basins
lai_months = exact_extract(lai.load(), basins, ["mean"], progress=False, output="pandas")
basins["lai_max"]  = lai_months.max(axis=1).to_numpy()
basins["lai_diff"] = (lai_months.max(axis=1) - lai_months.min(axis=1)).to_numpy()

In [ ]:
# Save to file
basins.to_file('data/hydro/basins_hydroATLAS_lev08.gpkg')